Dataset Link : https://www.kaggle.com/datasets/lainguyn123/student-performance-factors

In [2]:
from google.colab import files
uploaded = files.upload()


Saving dataset.csv to dataset.csv


In [3]:
import pandas as pd
df = pd.read_csv('dataset.csv')

In [4]:
print(df.head(5))

   Hours_Studied  Attendance Parental_Involvement Access_to_Resources  \
0             23          84                  Low                High   
1             19          64                  Low              Medium   
2             24          98               Medium              Medium   
3             29          89                  Low              Medium   
4             19          92               Medium              Medium   

  Extracurricular_Activities  Sleep_Hours  Previous_Scores Motivation_Level  \
0                         No            7               73              Low   
1                         No            8               59              Low   
2                        Yes            7               91           Medium   
3                        Yes            8               98           Medium   
4                        Yes            6               65           Medium   

  Internet_Access  Tutoring_Sessions Family_Income Teacher_Quality  \
0             Ye

In [5]:
df = df.select_dtypes(include = ['number'])

In [6]:
print(df.head())

   Hours_Studied  Attendance  Sleep_Hours  Previous_Scores  Tutoring_Sessions  \
0             23          84            7               73                  0   
1             19          64            8               59                  2   
2             24          98            7               91                  2   
3             29          89            8               98                  1   
4             19          92            6               65                  3   

   Physical_Activity  Exam_Score  
0                  3          67  
1                  4          61  
2                  4          74  
3                  4          71  
4                  4          70  


In [7]:
print(df.isnull().sum())

Hours_Studied        0
Attendance           0
Sleep_Hours          0
Previous_Scores      0
Tutoring_Sessions    0
Physical_Activity    0
Exam_Score           0
dtype: int64


In [13]:
from sklearn.model_selection import train_test_split
X = df.drop(columns = ['Exam_Score'])
Y = df['Exam_Score']
X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)


(5285, 6)
(1322, 6)
(5285,)
(1322,)


Feature Scaling

In [14]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

Build a Neural Network

In [15]:
import torch
import torch.nn as nn

In [27]:
class NeuralNetwork(nn.Module):
    def __init__(self, input_features):
        super().__init__()

        self.layers = nn.ModuleList([
            nn.Linear(input_features, 4),
            nn.Linear(4, 4)
        ])

        self.output = nn.Linear(4, 1)

    def forward(self, x):
        for layer in self.layers:
            x = torch.relu(layer(x))

        x = self.output(x)
        return x

In [28]:
model = NeuralNetwork(X_train.shape[1])

In [29]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters() , lr = 0.001)

In [38]:
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

In [43]:
for epoch in range(10000):
    predictions = model(X_train)
    loss = criterion(predictions, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 1000 == 0:
        print(f"Epoch: {epoch + 1}, Loss: {loss.item():.4f}")

Epoch: 1000, Loss: 6.2026
Epoch: 2000, Loss: 6.2020
Epoch: 3000, Loss: 6.2013
Epoch: 4000, Loss: 6.2006
Epoch: 5000, Loss: 6.2000
Epoch: 6000, Loss: 6.1995
Epoch: 7000, Loss: 6.1987
Epoch: 8000, Loss: 6.1981
Epoch: 9000, Loss: 6.1975
Epoch: 10000, Loss: 6.1969


In [44]:
model.eval()

with torch.no_grad():
    predictions = model(X_test)

    actual = y_test.numpy()
    predicted = predictions.numpy()

    error = abs(actual - predicted)

    accuracy_5 = (error <= 5).mean() * 100

print(f"Accuracy (within 5 marks): {accuracy_5:.2f}%")

Accuracy (within 5 marks): 99.32%


In [45]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

model.eval()

with torch.no_grad():
    predictions = model(X_test)

actual = y_test.cpu().numpy().reshape(-1)
predicted = predictions.cpu().numpy().reshape(-1)

mae = mean_absolute_error(actual, predicted)
rmse = np.sqrt(mean_squared_error(actual, predicted))
r2 = r2_score(actual, predicted)

accuracy_5 = (np.abs(actual - predicted) <= 5).mean() * 100

print(f"Accuracy (within 5 marks): {accuracy_5:.2f}%")
print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R2 Score: {r2:.4f}")

Accuracy (within 5 marks): 99.32%
MAE: 1.28
RMSE: 2.27
R2 Score: 0.6340
